# Ejemplos base de carga de datos y ejecucion cuantica

Este notebook reune ejemplos minimos para iniciar los challenges del repositorio: lectura de un Excel con `pandas` y pruebas de ejecucion de circuitos cuanticos con PennyLane, Qiskit Aer y CUDA Quantum.

## 1. Circuito cuantico con PennyLane

Construye un circuito parametrico pequeno, hace una ejecucion de calentamiento y mide el tiempo de una segunda ejecucion. Si no tienes GPU o `lightning.gpu`, cambia el dispositivo por `default.qubit` o `lightning.qubit`.

In [ ]:
##########################################
## 2. Circuito cuántico con Pennylane ##
##########################################


import pennylane as qml
import numpy as np
import time

n_qubits = 4
depth = 3
## Devices: default.qubit lightning.gpu lightning.qubit

dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qnode(dev)
def circuit(params):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for d in range(depth):
        for i in range(n_qubits):
            qml.RX(params[d, i, 0], wires=i)
            qml.RY(params[d, i, 1], wires=i)
            qml.RZ(params[d, i, 2], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    return qml.expval(qml.PauliZ(0))

params = np.random.random((depth, n_qubits, 3))

# warmup
circuit(params)

t0 = time.perf_counter()
result = circuit(params)
t1 = time.perf_counter()

print("Resultado:", result)
print("Tiempo:", t1 - t0, "s")


## 2. Circuito cuantico con Qiskit Aer

Crea el mismo patron de circuito con Qiskit, lo transpila para `AerSimulator` y mide el tiempo de simulacion usando `statevector`. Si tu instalacion no expone GPU, cambia `device="GPU"` por `device="CPU"`.

In [ ]:
###########################################
## 3. Circuito cuántico con Qiskit ##
############################################
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import time

n_qubits = 4
depth = 3

qc = QuantumCircuit(n_qubits)

for i in range(n_qubits):
    qc.h(i)

for d in range(depth):
    for i in range(n_qubits):
        qc.rx(0.123, i)
        qc.ry(0.456, i)
        qc.rz(0.789, i)

    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)

qc.save_statevector()

backend = AerSimulator(
    method="statevector",
    device="GPU"
)

print("Devices disponibles:", backend.available_devices())

tqc = transpile(qc, backend)

# warmup
backend.run(tqc).result()

t0 = time.perf_counter()
result = backend.run(tqc).result()
t1 = time.perf_counter()

state = result.get_statevector()

print("Statevector size:", len(state))
print("Tiempo:", t1 - t0, "s")

## 3. Circuito cuantico con CUDA Quantum

Ejemplo equivalente usando `cudaq` con target NVIDIA. Requiere una instalacion de CUDA Quantum compatible con la GPU disponible.

In [ ]:
############################################
## 4. Circuito cuántico con CUDA Quantum ##
############################################
import cudaq
import time

n_qubits = 4
depth = 3

cudaq.set_target("nvidia")

@cudaq.kernel
def circuit():
    q = cudaq.qvector(n_qubits)

    for i in range(n_qubits):
        h(q[i])

    for d in range(depth):
        for i in range(n_qubits):
            rx(0.123, q[i])
            ry(0.456, q[i])
            rz(0.789, q[i])

        for i in range(n_qubits - 1):
            x.ctrl(q[i], q[i + 1])

    mz(q)

# warmup
cudaq.sample(circuit, shots_count=100)

t0 = time.perf_counter()
result = cudaq.sample(circuit, shots_count=1000)
t1 = time.perf_counter()

print(result)
print("Tiempo:", t1 - t0, "s")